# Business Analytics 2, Week 4: Data Preparation

Companion notebook for the Week 4 page of the BA2 course app: profiling, cleaning, integrating and transforming the messy jam stand export.

**Run the cells in order**: the first creates `df` (the raw export), and later cells build `clean` from it. If you restart the runtime, run them again from the top.

_Generated from the code blocks on the Week 4 page._

## 1. Load the raw May export

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "id":          range(1, 16),
    "flavor":      ["Strawberry", "blueberry", "Peach", "Strawberry ", "Blueberry", "Peach",
                    "Strawberry", "Blueberry", "Peach", "Strawberry", "Blueberry", "Peach",
                    "Lemon", "Strawberry", "Peach"],
    "market_date": ["2024-05-04", "2024-05-04", "04/05/2024", "2024-05-11", "2024-05-11", "2024-05-11",
                    "2024-05-18", "2024-05-18", "2024-05-18", "2024-05-25", "2024-05-25", "2024-05-25",
                    "2024-05-25", "2024-05-25", "2024-01-01"],
    "units_sold":  [22, 15, 10, 25, None, 8, 180, 17, -12, 35, 16, 10, 9, 35, 0],
})
df

## 2. Profile it

In [ ]:
df.info()                          # units_sold: 14 non-null out of 15
print(df.isna().sum())             # one missing units_sold
print(df["flavor"].value_counts()) # 6 spellings for 4 flavors
print(df["units_sold"].describe()) # min -12, max 180: both suspicious

## 3. Find rows that break a rule

In [ ]:
bad_units = df[df["units_sold"].isna() | (df["units_sold"] < 0)]
bad_dates = df[~df["market_date"].str.match(r"^\d{4}-\d{2}-\d{2}$")]
key = df["flavor"].str.strip().str.title()
dupes = df[df.assign(flavor=key).duplicated(subset=["flavor", "market_date", "units_sold"], keep=False)]
bad_units, bad_dates, dupes

## 4. Clean a copy

In [ ]:
clean = df.copy()                                   # never edit the raw data
clean["flavor"] = clean["flavor"].str.strip().str.title()
clean.loc[clean["id"] == 3, "market_date"] = "2024-05-04"
clean.loc[clean["id"] == 5, "units_sold"] = 12        # from the paper log
clean.loc[clean["id"] == 7, "units_sold"] = 18        # typo, confirmed by the owner
clean.loc[clean["id"] == 9, "units_sold"] = 12        # sign error
clean = clean[~clean["id"].isin([14, 15])]            # duplicate; till test row
clean["market_date"] = pd.to_datetime(clean["market_date"])
clean["units_sold"] = clean["units_sold"].astype(int)
print(len(clean), clean["units_sold"].sum())          # 13 209

## 5. Integrate the supplier prices

In [ ]:
prices = pd.DataFrame({
    "flavour_name": ["STRAWBERRY", "Blue berry", "Peach", "Lemon"],
    "price_cents":  [500, 600, 400, 450],
})
prices["key"] = prices["flavour_name"].str.replace(" ", "").str.upper()
clean["key"] = clean["flavor"].str.replace(" ", "").str.upper()

merged = clean.merge(prices[["key", "price_cents"]], on="key", how="left")
merged["price"] = merged["price_cents"] / 100          # cents -> dollars
merged["revenue"] = merged["units_sold"] * merged["price"]
merged.groupby("flavor")[["units_sold", "revenue"]].sum()

## 6. Normalize the monthly totals

In [ ]:
monthly = pd.DataFrame({
    "month": ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"],
    "total": [16, 19, 27, 42, 66, 92, 108, 108, 74, 44, 25, 28],
})
t = monthly["total"]
monthly["min_max"] = ((t - t.min()) / (t.max() - t.min())).round(2)
monthly["z_score"] = ((t - t.mean()) / t.std()).round(2)   # .std() is the sample standard deviation
monthly["decimal"] = t / 1000                                # largest value 108 has 3 digits
monthly

## 7. Discretize and roll up

In [ ]:
clean["day_type"] = pd.cut(clean["units_sold"], bins=[0, 11, 24, float("inf")],
                           labels=["slow", "normal", "busy"])
print(clean["day_type"].value_counts())   # normal 7, slow 4, busy 2

family = {"Strawberry": "berry", "Blueberry": "berry", "Peach": "stone fruit", "Lemon": "citrus"}
clean["family"] = clean["flavor"].map(family)
clean.groupby("family")["units_sold"].sum()   # berry 160, stone fruit 40, citrus 9